# Fako Online - Full Pipeline Server (Kaggle)

**Bark TTS + SadTalker + Easy-Wav2Lip**

This notebook runs all three models on a single Kaggle GPU session.

### Instructions
1. Enable **GPU T4 x2** (Settings > Accelerator)
2. Add datasets: `kingtechie/bark-model`, `kingtechie/sadtalker-model`, `kingtechie/wav2lip-model`
3. Run all cells in order
4. The server will start on port 8000

In [ ]:
# Cell 1: Setup paths
import os

WORKING_DIR = "/kaggle/working/outputs"
os.makedirs(WORKING_DIR, exist_ok=True)

BARK_DIR = "/kaggle/input/bark-model"
SADTALKER_DIR = "/kaggle/input/sadtalker-model"
WAV2LIP_DIR = "/kaggle/input/wav2lip-model"

print(f"Bark: {BARK_DIR}")
print(f"SadTalker: {SADTALKER_DIR}")
print(f"Easy-Wav2Lip: {WAV2LIP_DIR}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q fastapi uvicorn python-multipart
!pip install -q transformers scipy
!pip install -q gfpgan
!pip install -q librosa
print("Dependencies installed!")

In [ ]:
# Cell 3: Import libraries
import torch
import numpy as np
import io
import base64
import subprocess
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 4: Load SadTalker
import sys
sys.path.insert(0, SADTALKER_DIR)

from src.gradio_demo import SadTalker

sadtalker = SadTalker(
    checkpoint_path=f"{SADTALKER_DIR}/checkpoints",
    config_path=f"{SADTALKER_DIR}/src/config",
    lazy_dir=f"{SADTALKER_DIR}/gfpgan/weights"
)
print("SadTalker loaded!")

In [ ]:
# Cell 5: Load Easy-Wav2Lip
print("Easy-Wav2Lip ready to load on first use")

In [ ]:
# Cell 6: Load Bark TTS
from transformers import AutoProcessor, BarkModel

processor = AutoProcessor.from_pretrained(BARK_DIR)
model = BarkModel.from_pretrained(BARK_DIR)
model.to("cpu")

print("Bark TTS loaded!")

In [ ]:
# Cell 7: Generation functions
import soundfile as sf

def generate_tts(text, voice_preset="v2/en_speaker_6"):
    inputs = processor(text, voice_preset=voice_preset, return_tensors="pt")
    inputs = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in inputs.items()}
    model.to("cuda")
    with torch.no_grad():
        audio_values = model.generate(**inputs, do_sample=True)
    audio = audio_values.cpu().numpy().squeeze()
    model.to("cpu")
    torch.cuda.empty_cache()
    output_path = f"{WORKING_DIR}/emotional_speech.wav"
    sf.write(output_path, audio, model.generation_config.sample_rate)
    return output_path

def generate_avatar(image_path, audio_path, result_filename="talking-head.mp4"):
    result = sadtalker.test(
        image_path=image_path,
        audio_path=audio_path,
        result_dir=WORKING_DIR,
        still_mode=False,
        use_enhancer=True,
        batch_size=2,
        size=256,
        pose_style=0
    )
    return result["mp4_path"] if isinstance(result, dict) else result

def run_wav2lip(video_path, audio_path):
    output_path = f"{WORKING_DIR}/refined_output.mp4"
    cmd = [
        "python", f"{WAV2LIP_DIR}/inference.py",
        "--checkpoint_path", f"{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth",
        "--face", video_path,
        "--audio", audio_path,
        "--outfile", output_path,
        "--nosmooth"
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    return output_path

print("Generation functions defined!")

In [ ]:
# Cell 8: FastAPI server
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title="Fako Online - Full Pipeline API")

@app.get("/health")
async def health():
    return {"status": "ok", "models": ["bark", "sadtalker", "wav2lip"]}

@app.post("/generate-tts")
async def api_generate_tts(text: str = Form(...), voice_preset: str = Form("v2/en_speaker_6")):
    try:
        audio_path = generate_tts(text, voice_preset)
        return FileResponse(audio_path, media_type="audio/wav")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/generate-avatar")
async def api_generate_avatar(image: UploadFile = File(...), audio: UploadFile = File(...)):
    try:
        image_path = f"{WORKING_DIR}/{image.filename}"
        audio_path = f"{WORKING_DIR}/{audio.filename}"
        with open(image_path, "wb") as f:
            f.write(await image.read())
        with open(audio_path, "wb") as f:
            f.write(await audio.read())
        video_path = generate_avatar(image_path, audio_path)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/generate-full")
async def api_generate_full(
    image: UploadFile = File(...),
    text: str = Form(...),
    voice_preset: str = Form("v2/en_speaker_6"),
    refine_lips: bool = Form(True)
):
    try:
        image_path = f"{WORKING_DIR}/{image.filename}"
        with open(image_path, "wb") as f:
            f.write(await image.read())
        audio_path = generate_tts(text, voice_preset)
        video_path = generate_avatar(image_path, audio_path)
        if refine_lips:
            video_path = run_wav2lip(video_path, audio_path)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

print("FastAPI server defined!")

In [ ]:
# Cell 9: Start server
print("Starting server on port 8000...")
uvicorn.run(app, host="0.0.0.0", port=8000)